In [9]:
path_to_pkl = "/home/pablo.canosa/ssd/code_tests/Mask2formerCleanRepo/Mask2Former/backbone_weights/swin_tiny_patch4_window7_224.pkl"

# Print the content of the .pkl file
import pickle
with open(path_to_pkl, "rb") as f:
    data = pickle.load(f)

# print each layer and its shape
for layer_name, weights in data["model"].items():
    print(f"Layer: {layer_name}", f"Shape: {weights.shape}")



Layer: patch_embed.proj.weight Shape: torch.Size([96, 3, 4, 4])
Layer: patch_embed.proj.bias Shape: torch.Size([96])
Layer: patch_embed.norm.weight Shape: torch.Size([96])
Layer: patch_embed.norm.bias Shape: torch.Size([96])
Layer: layers.0.blocks.0.norm1.weight Shape: torch.Size([96])
Layer: layers.0.blocks.0.norm1.bias Shape: torch.Size([96])
Layer: layers.0.blocks.0.attn.qkv.weight Shape: torch.Size([288, 96])
Layer: layers.0.blocks.0.attn.qkv.bias Shape: torch.Size([288])
Layer: layers.0.blocks.0.attn.proj.weight Shape: torch.Size([96, 96])
Layer: layers.0.blocks.0.attn.proj.bias Shape: torch.Size([96])
Layer: layers.0.blocks.0.norm2.weight Shape: torch.Size([96])
Layer: layers.0.blocks.0.norm2.bias Shape: torch.Size([96])
Layer: layers.0.blocks.0.mlp.fc1.weight Shape: torch.Size([384, 96])
Layer: layers.0.blocks.0.mlp.fc1.bias Shape: torch.Size([384])
Layer: layers.0.blocks.0.mlp.fc2.weight Shape: torch.Size([96, 384])
Layer: layers.0.blocks.0.mlp.fc2.bias Shape: torch.Size([96])


In [10]:
import pickle
import torch
import numpy as np

path_to_pkl = "/home/pablo.canosa/ssd/code_tests/Mask2formerCleanRepo/Mask2Former/backbone_weights/swin_tiny_patch4_window7_224.pkl"
output_path = path_to_pkl.replace(".pkl", "_5ch_Rinit.pkl")

# Load pkl
with open(path_to_pkl, "rb") as f:
    data = pickle.load(f)

model = data["model"]

# Inspect original shape
w = model["patch_embed.proj.weight"]  # [96, 3, 4, 4]

# Convert to torch if needed (Detectron2 pkls are usually numpy)
if isinstance(w, np.ndarray):
    w = torch.from_numpy(w)

print("Original shape:", w.shape)

# --- Extract channels ---
w_r = w[:, 0:1, :, :]  # Red channel

# --- Build new weight tensor ---
# [R, G, B, RedEdge, NIR]
w_new = torch.cat([
    w,      # original RGB
    w_r,    # RedEdge ← R
    w_r     # NIR ← R
], dim=1)

print("New shape:", w_new.shape)

# Convert back to numpy (important for Detectron2 compatibility)
model["patch_embed.proj.weight"] = w_new.numpy()

# Save new pkl
with open(output_path, "wb") as f:
    pickle.dump(data, f)

print(f"Saved modified weights to: {output_path}")

Original shape: torch.Size([96, 3, 4, 4])
New shape: torch.Size([96, 5, 4, 4])
Saved modified weights to: /home/pablo.canosa/ssd/code_tests/Mask2formerCleanRepo/Mask2Former/backbone_weights/swin_tiny_patch4_window7_224_5ch_Rinit.pkl


In [11]:
import pickle
import torch
import numpy as np

path_to_pkl = "/home/pablo.canosa/ssd/code_tests/Mask2formerCleanRepo/Mask2Former/backbone_weights/swin_tiny_patch4_window7_224.pkl"
output_path = path_to_pkl.replace(".pkl", "_5ch_AVGinit.pkl")

# Load pkl
with open(path_to_pkl, "rb") as f:
    data = pickle.load(f)

model = data["model"]

# Inspect original shape
w = model["patch_embed.proj.weight"]  # [96, 3, 4, 4]

# Convert to torch if needed (Detectron2 pkls are usually numpy)
if isinstance(w, np.ndarray):
    w = torch.from_numpy(w)

print("Original shape:", w.shape)

# --- Extract channels ---
w_r = w[:, 0:1, :, :]  # Red channel
w_g = w[:, 1:2, :, :]  # Green channel
w_b = w[:, 2:3, :, :]  # Blue channel

# Average of RGB for new channels
w_avg = (w_r + w_g + w_b) / 3.0

# --- Build new weight tensor ---
# [R, G, B, RedEdge, NIR]
w_new = torch.cat([
    w,      # original RGB
    w_avg,    # RedEdge ← R
    w_avg     # NIR ← R
], dim=1)

print("New shape:", w_new.shape)

# Convert back to numpy (important for Detectron2 compatibility)
model["patch_embed.proj.weight"] = w_new.numpy()

# Save new pkl
with open(output_path, "wb") as f:
    pickle.dump(data, f)

print(f"Saved modified weights to: {output_path}")

Original shape: torch.Size([96, 3, 4, 4])
New shape: torch.Size([96, 5, 4, 4])
Saved modified weights to: /home/pablo.canosa/ssd/code_tests/Mask2formerCleanRepo/Mask2Former/backbone_weights/swin_tiny_patch4_window7_224_5ch_AVGinit.pkl
